In [1]:

import pandas as pd 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv(r"D:\Tarun's code\mini Project(MRS)\MOVIE-RECOMMENDATION-SYSTEM\LargeSet\movies.csv")

print(df.head())
print(df.info())
print(df.shape)

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86537 entries, 0 to 86536
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  86537 non-null  int64 
 1   title    86537 non-null  object
 2   genres   86537 non-null  object
dtypes: int64(1), object(2)
memory usage: 2.0+ MB
None
(86537, 3)


In [2]:
movies = pd.read_csv(r"smallset\movies.csv")
ratings = pd.read_csv(r"smallset\ratings.csv")
print(ratings.head())
print(movies.head())

   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


In [3]:
movie_stats = ratings.groupby('movieId')['rating'].agg(['mean', 'count'])
print(movie_stats.head())

movie_stats = movie_stats.join(movies.set_index('movieId'), on='movieId')
print(movie_stats.head())


             mean  count
movieId                 
1        3.920930    215
2        3.431818    110
3        3.259615     52
4        2.357143      7
5        3.071429     49
             mean  count                               title  \
movieId                                                        
1        3.920930    215                    Toy Story (1995)   
2        3.431818    110                      Jumanji (1995)   
3        3.259615     52             Grumpier Old Men (1995)   
4        2.357143      7            Waiting to Exhale (1995)   
5        3.071429     49  Father of the Bride Part II (1995)   

                                              genres  
movieId                                               
1        Adventure|Animation|Children|Comedy|Fantasy  
2                         Adventure|Children|Fantasy  
3                                     Comedy|Romance  
4                               Comedy|Drama|Romance  
5                                             

In [4]:
# replacing '|' with space in genres column

movies["genres"] = movies["genres"].str.replace("|", " ")



In [5]:
popular_movies = movie_stats[movie_stats['count'] >= 50]
print(popular_movies.sort_values('mean', ascending=False).head(10))

             mean  count                                              title  \
movieId                                                                       
318      4.429022    317                   Shawshank Redemption, The (1994)   
858      4.289062    192                              Godfather, The (1972)   
2959     4.272936    218                                  Fight Club (1999)   
1276     4.271930     57                              Cool Hand Luke (1967)   
750      4.268041     97  Dr. Strangelove or: How I Learned to Stop Worr...   
904      4.261905     84                                 Rear Window (1954)   
1221     4.259690    129                     Godfather: Part II, The (1974)   
48516    4.252336    107                               Departed, The (2006)   
1213     4.250000    126                                  Goodfellas (1990)   
912      4.240000    100                                  Casablanca (1942)   

                              genres  
movieId     

In [6]:
tfidf = TfidfVectorizer(stop_words="english") #removing english stop words like is , am , the etc
tfidf = tfidf.fit_transform(movies['genres']) #fitting and transforming the genres column

# print(tfidf)

In [7]:
#calculating cosine similarity matrix

similarity = cosine_similarity(tfidf)
print(similarity)

[[1.         0.81357774 0.15276924 ... 0.         0.4210373  0.26758648]
 [0.81357774 1.         0.         ... 0.         0.         0.        ]
 [0.15276924 0.         1.         ... 0.         0.         0.57091541]
 ...
 [0.         0.         0.         ... 1.         0.         0.        ]
 [0.4210373  0.         0.         ... 0.         1.         0.        ]
 [0.26758648 0.         0.57091541 ... 0.         0.         1.        ]]


In [8]:
# # title = input("enter the movie name: ")
# print(title)

movies["title_lower"] = movies["title"].str.lower()
indices = pd.Series(movies.index, index=movies["title_lower"]).drop_duplicates()
print(indices.head())


title_lower
toy story (1995)                      0
jumanji (1995)                        1
grumpier old men (1995)               2
waiting to exhale (1995)              3
father of the bride part ii (1995)    4
dtype: int64


In [9]:
# idx_movie = pd.Series(movies.index, index=movies['title'].str[:-7].str.lower()).drop_duplicates()
# print(idx_movie.head())

# type(idx_movie)


In [10]:
def recommend2(title, similarity=similarity):
    try:
     idx = idx_movie[title]
    except KeyError:
        print("Movie not found. Please check the title and try again.")
        exit()
    similarity_score = list(enumerate(similarity[idx]))

     sorting the movies based on similarity score
    similar_movies = sorted(similarity_score, key=lambda x: x[1], reverse=True)

     print(similar_movies)
    
    similar_movies = similar_movies[:11] # excluding first movie as it is the same movie
    movie_indices = [i[0] for i in similar_movies]
    print(movies['title'].iloc[movie_indices])

recommend(title)

IndentationError: unexpected indent (1299803922.py, line 9)

In [ ]:
def recommend(self, title, num=10, min_count=50):
    self.title = title
    title = str(title).lower()

    # 1. Check if movie exists
    if title not in indices:
        print(f"Movie '{title}' not found in dataset.")
        return

    # 2. Get index of the movie in 'movies' DataFrame
    idx = indices[title]

    # get the similarity scores
    similarity_score = list(enumerate(similarity[idx]))

    # sorting the movies based on similarity score
    similarity_score = sorted(similarity_score, key=lambda x: x[1], reverse=True)

    # filtering by Ratings
    filtered = []

    for i, score in similarity_score:
        movie_id = movies.loc[i, "movieId"]

        # Get stats (mean rating & count) from movie_stats using movieId
        if movie_id in movie_stats.index:
            avg_rating = movie_stats.loc[movie_id, "mean"]
            rating_count = movie_stats.loc[movie_id, "count"]
        else:
            continue  # if somehow missing, skip

        if rating_count >= min_count:
            filtered.append((i, score, avg_rating, rating_count))

        if len(filtered) >= num:
            break

    results = pd.DataFrame(
        {
            "title": movies.loc[i, "title"],
            "genres": movies.loc[i, "genres"],
            "similarity": round(score, 3),
            "avg_rating": round(avg_rating, 2),
            "rating_count": int(rating_count),
        }
        for (i, score, avg_rating, rating_count) in filtered
    )

    return results

recommend("fight club")


TypeError: recommend() missing 1 required positional argument: 'title'

In [ ]:
# Streamlit Ui

# Streamlit UI
# -------------------------------
st.title("🎬 Movie Recommendation System")
st.write("Type a movie name and get similar movie recommendations based on genres.")

user_input = st.text_input("Enter a movie title:", "")

if st.button("Recommend"):
    if user_input.strip() == "":
        st.warning("Please enter a movie name!")
    else:
        results = recommend(user_input)
        
        if results is None:
            st.error("Movie not found. Check spelling or try another movie.")
        else:
            st.success(f"Top Recommendations for: {user_input.title()}")
            st.table(results)
            



2025-11-25 15:44:48.944 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-25 15:44:49.615 
  command:

    streamlit run d:\Tarun's code\mini Project(MRS)\MOVIE-RECOMMENDATION-SYSTEM\.venv311\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2025-11-25 15:44:49.615 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-25 15:44:49.615 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-25 15:44:49.615 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-25 15:44:49.615 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-25 15:44:49.615 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2

In [ ]:
# list of similarity score 
# try:
#  idx = idx_movie[title]
# except KeyError:
#     print("Movie not found. Please check the title and try again.")
#     exit()
# similarity_score = list(enumerate(similarity[idx]))

# # sorting the movies based on similarity score
# similar_movies = sorted(similarity_score, key=lambda x: x[1], reverse=True)

# print(similar_movies)

In [ ]:
# TAKING A MOVIE TITLE AS INPUT AND RECOMMENDING SIMILAR MOVIES BASED ON GENRES

# def recommend_movies(movie_title):
#     #getting the index of the movie that matches the title
#     movie_idx = movies[movies['title'] == movie_title].index[0]
#     #getting the pairwise similarity scores of all movies with that movie
#     sim_scores = list(enumerate(similarity[movie_idx]))
#     #sorting the movies based on similarity scores
#     sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
#     #getting the scores of the 10 most similar movies
#     sim_scores = sim_scores[1:11]
#     #getting the movie indices
#     movie_indices = [i[0] for i in sim_scores]
#     #returning the top 10 most similar movies
#     return movies['title'].iloc[movie_indices]

# print(recommend_movies("Toy Story (1995)"))

In [ ]:
import pandas as pd

netflix = pd.read_csv("netflix_titles.csv")
movies = pd.read_csv(r"smallset\movies.csv")
print(netflix.head())

  show_id     type                  title         director  \
0      s1    Movie   Dick Johnson Is Dead  Kirsten Johnson   
1      s2  TV Show          Blood & Water              NaN   
2      s3  TV Show              Ganglands  Julien Leclercq   
3      s4  TV Show  Jailbirds New Orleans              NaN   
4      s5  TV Show           Kota Factory              NaN   

                                                cast        country  \
0                                                NaN  United States   
1  Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...   South Africa   
2  Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...            NaN   
3                                                NaN            NaN   
4  Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...          India   

  date_added  release_year rating   duration  \
0  25-Sep-21          2020  PG-13     90 min   
1  24-Sep-21          2021  TV-MA  2 Seasons   
2  24-Sep-21          2021  TV-MA   1 Season   
3  24-Se

In [ ]:
for col in ["listed_in", "description", "cast", "director"]:
    if col in netflix.columns:
        netflix[col] = netflix[col].fillna("")
    else:
        netflix[col] = ""

In [ ]:
netflix["text"] = (
    netflix["listed_in"] + " " +
    netflix["description"] + " " +
    netflix["cast"] + " " +
    netflix["director"]
)
netflix["listed_in"] = netflix["listed_in"].fillna("")
netflix["description"] = netflix["description"].fillna("")
netflix["cast"] = netflix["cast"].fillna("")
netflix["director"] = netflix["director"].fillna("")

In [ ]:
movies["source"] = "movielens"
movies["display_genre"] = movies["genres"]
movies["display_type"] = "Movie"

movies2 = movies[["title", "title_lower", "text", "display_genre", "display_type", "source"]]

netflix["source"] = "netflix"
netflix["display_genre"] = netflix["listed_in"]
# netflix has its own 'type' column (Movie / TV Show)
netflix["display_type"] = netflix["type"]

netflix2 = netflix[["title", "title_lower", "text", "display_genre", "display_type", "source"]]

KeyError: "['title_lower', 'text'] not in index"